#Question 4

In [ ]:
# Import relevent libraries
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
#Load the Dataset
df = pd.read_csv("AMT Anomaly Dataset.csv")
FEATURES = ["delay_ms", "clutch_temp_c", "torque_var_pct", "rpm_diff"]
X = df[FEATURES].values
y_sev = df["severity"].values
y_act = df["action"].values
df.head()

,id,delay_ms,clutch_temp_c,torque_var_pct,rpm_diff,severity,action
0,1,72,86,2.1,45,Normal,NO_ACTION
1,2,88,92,3.8,62,Normal,NO_ACTION
2,3,95,98,4.9,79,Normal,NO_ACTION
3,4,66,90,1.7,34,Normal,NO_ACTION
4,5,101,94,5.0,80,Normal,NO_ACTION


In [ ]:
# Machine Learning Based Model

le_sev = LabelEncoder()
le_act = LabelEncoder()
y_sev_enc = le_sev.fit_transform(y_sev)
y_act_enc = le_act.fit_transform(y_act)

# Split the dataset. We will use X_te(X test data) for evaluating all three models.
X_tr, X_te, ys_tr, ys_te, ya_tr, ya_te = train_test_split(
    X, y_sev_enc, y_act_enc, test_size=0.25, random_state=42, stratify=y_sev_enc)

clf_sev = RandomForestClassifier(n_estimators=100, random_state=42)
clf_act = RandomForestClassifier(n_estimators=100, random_state=42)
clf_sev.fit(X_tr, ys_tr)
clf_act.fit(X_tr, ya_tr)

def ml_detect(delay_ms, clutch_temp_c, torque_var_pct, rpm_diff):
    row = np.array([[delay_ms, clutch_temp_c, torque_var_pct, rpm_diff]])
    sev = le_sev.inverse_transform(clf_sev.predict(row))[0]
    act = le_act.inverse_transform(clf_act.predict(row))[0]
    return sev, act

In [ ]:
#Fuzzy Logic Based System

def _tri(x, a, b, c):
    #Triangular membership function
    if x <= a or x >= c: return 0.0
    if x <= b: return (x - a) / (b - a)
    return (c - x) / (c - b)

def _trap(x, a, b, c, d):
    #Trapezoidal membership function
    if x <= a or x >= d: return 0.0
    if b <= x <= c: return 1.0
    if x < b: return (x - a) / (b - a)
    return (d - x) / (d - c)

def fuzzy_logic_detect(delay_ms, clutch_temp_c, torque_var_pct, rpm_diff):
    #Mamdani-style fuzzy inference to output severity and action

    # Fuzzify inputs
    d_low  = _trap(delay_ms,  60, 60, 95, 115)
    d_med  = _tri (delay_ms,  95, 130, 165)
    d_high = _trap(delay_ms, 145, 165, 200, 200)

    t_low  = _trap(clutch_temp_c,  80,  80, 97, 110)
    t_med  = _tri (clutch_temp_c,  97, 113, 128)
    t_high = _trap(clutch_temp_c, 120, 130, 145, 145)

    q_low  = _trap(torque_var_pct, 1, 1,  5,  7)
    q_med  = _tri (torque_var_pct, 5, 8, 11)
    q_high = _trap(torque_var_pct, 9, 11, 15, 15)

    r_low  = _trap(rpm_diff,  30,  30,  80, 100)
    r_med  = _tri (rpm_diff,  80, 130, 170)
    r_high = _trap(rpm_diff, 155, 175, 220, 220)

    # Fuzzy rules for severity classes
    rule_normal   = max(min(d_low,  t_low,  q_low,  r_low),
                        min(d_low,  t_low,  q_low,  r_med))
    rule_moderate = max(min(d_med,  t_low,  q_low,  r_low),
                        min(d_low,  t_med,  q_low,  r_low),
                        min(d_low,  t_low,  q_med,  r_low),
                        min(d_low,  t_low,  q_low,  r_med),
                        min(d_med,  t_med,  q_med,  r_med))
    rule_severe   = max(min(d_high, t_high, q_high, r_high),
                        min(d_high, t_high, q_med,  r_med),
                        min(d_med,  t_high, q_high, r_high),
                        min(d_high, q_high),
                        min(t_high, r_high))

    # Crisp severity via centroid
    num = (0 * rule_normal + 5 * rule_moderate + 10 * rule_severe)
    den = (rule_normal + rule_moderate + rule_severe + 1e-9)
    score = num / den

    if score < 3.0: severity = "Normal"
    elif score < 7.0: severity = "Moderate"
    else: severity = "Severe"

    # Action rules
    if severity == "Normal": return severity, "NO_ACTION"
    if t_high > 0.4: action = "CLUTCH_PRESSURE_RECALIB"
    elif r_high > 0.4: action = "RPM_SYNC"
    elif q_high > 0.4 or q_med > 0.5: action = "TORQUE_REDISTRIBUTION"
    else: action = "SHIFT_TIMING_ADJUST"

    return severity, action

In [ ]:
#Rule Based System

def rule_based_detect(delay_ms, clutch_temp_c, torque_var_pct, rpm_diff):
    #Expert threshold rules to determine severity and action
    # Severity
    if (delay_ms >= 160 or clutch_temp_c >= 125 or
            torque_var_pct >= 11.0 or rpm_diff >= 170):
        severity = "Severe"
    elif (delay_ms >= 110 or clutch_temp_c >= 105 or
          torque_var_pct >= 6.0 or rpm_diff >= 90):
        severity = "Moderate"
    else:
        severity = "Normal"

    # Action
    if severity == "Normal":
        return severity, "NO_ACTION"

    if clutch_temp_c >= 125:
        action = "CLUTCH_PRESSURE_RECALIB" # Represents Clutch Pressure Recalibration
    elif rpm_diff >= 170:
        action = "RPM_SYNC" # Represents Engine RPM Synchronization
    elif torque_var_pct >= 6.0:
        action = "TORQUE_REDISTRIBUTION" # Represents Torque Redistribution
    else:
        action = "SHIFT_TIMING_ADJUST" # Represents Adaptive Gear Shift Timing Adjustment

    return severity, action

In [44]:
import time
import numpy as np
from sklearn.metrics import classification_report

# Evaluation Function
def evaluate_approach(name, detect_fn, X_data, y_sev_true, y_act_true):
    """
    Evaluates a specific model approach based on inference time and effectiveness.
    """
    sev_preds, act_preds = [], []

    # Comparison of Inference time
    start_time = time.perf_counter()
    for row in X_data:
        s, a = detect_fn(*row)
        sev_preds.append(s)
        act_preds.append(a)
    elapsed_time = time.perf_counter() - start_time

    sev_preds = np.array(sev_preds)
    act_preds = np.array(act_preds)

    # Assess Anomaly Mitigation Effectiveness [True Positive(TP) / True Negative(TN) / False Psitive(FP) / False Negative(FN)]

    # True Positive - Correctly identified an anomaly
    tp_sev = sum((sev_preds != "Normal") & (y_sev_true != "Normal"))

    # True Negative - Correctly identified normal behavior
    tn_sev = sum((sev_preds == "Normal") & (y_sev_true == "Normal"))

    # False Positive - Model predicted an anomaly (Moderate/Severe), but it was actually Normal.
    fp_sev = sum((sev_preds != "Normal") & (y_sev_true == "Normal"))

    # False Negative - Model predicted Normal, but there was actually an anomaly (Moderate/Severe).
    fn_sev = sum((sev_preds == "Normal") & (y_sev_true != "Normal"))

    # General Accuracy
    acc_sev = np.mean(sev_preds == y_sev_true)
    acc_act = np.mean(act_preds == y_act_true)

    print(f"\n")
    print(f"  APPROACH: {name}")
    print(f"  Total inference time : {elapsed_time*1000:.3f} ms")
    print(f"  Inference per sample : {elapsed_time/len(X_data)*1000:.5f} ms/sample")
    print(f"\n Effectiveness")
    print(f"  Severity Accuracy  : {acc_sev*100:.1f}%")
    print(f"  Action Accuracy    : {acc_act*100:.1f}%")
    print(f"  True Positives     : {tp_sev}")
    print(f"  True Negatives     : {tn_sev}")
    print(f"  False Positives    : {fp_sev}")
    print(f"  False Negatives    : {fn_sev}")

    return {
        "name": name,
        "time_ms": elapsed_time * 1000,
        "time_per_sample_ms": elapsed_time / len(X_data) * 1000,
        "sev_accuracy": acc_sev,
        "act_accuracy": acc_act,
        "true_positives": tp_sev,
        "true_negatives": tn_sev,
        "false_positives": fp_sev,
        "false_negatives": fn_sev,
    }

# Test Running
# Assuming X_te, y_sev_te_raw, y_act_te_raw are the test dataset splits
results = []

results.append(evaluate_approach("Rule-Based System", rule_based_detect, X_te, y_sev_te_raw, y_act_te_raw))
results.append(evaluate_approach("Fuzzy Logic System", fuzzy_logic_detect, X_te, y_sev_te_raw, y_act_te_raw))
results.append(evaluate_approach("ML (Random Forest)", ml_detect, X_te, y_sev_te_raw, y_act_te_raw))





  APPROACH: Rule-Based System
  Total inference time : 0.112 ms
  Inference per sample : 0.00748 ms/sample

 Effectiveness
  Severity Accuracy  : 100.0%
  Action Accuracy    : 73.3%
  True Positives     : 12
  True Negatives     : 3
  False Positives    : 0
  False Negatives    : 0


  APPROACH: Fuzzy Logic System
  Total inference time : 0.314 ms
  Inference per sample : 0.02092 ms/sample

 Effectiveness
  Severity Accuracy  : 60.0%
  Action Accuracy    : 46.7%
  True Positives     : 6
  True Negatives     : 3
  False Positives    : 0
  False Negatives    : 6


  APPROACH: ML (Random Forest)
  Total inference time : 394.750 ms
  Inference per sample : 26.31665 ms/sample

 Effectiveness
  Severity Accuracy  : 73.3%
  Action Accuracy    : 66.7%
  True Positives     : 12
  True Negatives     : 1
  False Positives    : 2
  False Negatives    : 0


In [49]:
#Comparision Summary

print("COMPARISON SUMMARY")
print(f"\n")
header = f"{'Approach':<22} {'ms/sample':>11} {'Sev Acc':>9} {'Act Acc':>9} {'TP':>5} {'TN':>5} {'FP':>5} {'FN':>5}"
print(header)
print("-" * len(header))
for r in results:
    print(f"{r['name']:<22} {r['time_per_sample_ms']:>11.5f} "
          f"{r['sev_accuracy']*100:>8.1f}% {r['act_accuracy']*100:>8.1f}% "
          f"{r['true_positives']:>5} {r['true_negatives']:>5} "
          f"{r['false_positives']:>5} {r['false_negatives']:>5}")

# Logic to help justify the best approach for real-time
best_speed = min(results, key=lambda x: x['time_per_sample_ms'])
best_safety = min(results, key=lambda x: x['false_negatives'])


COMPARISON SUMMARY


Approach                 ms/sample   Sev Acc   Act Acc    TP    TN    FP    FN
------------------------------------------------------------------------------
Rule-Based System          0.00748    100.0%     73.3%    12     3     0     0
Fuzzy Logic System         0.02092     60.0%     46.7%     6     3     0     6
ML (Random Forest)        26.31665     73.3%     66.7%    12     1     2     0
